# QUANT Classifier on 5 Standardized SWAN-SF Features

This notebook trains an **aeon `QUANTClassifier`** using only these five standardized SHARP/SWAN-SF features:

1. `TOTUSJH` — Total unsigned current helicity
2. `TOTBSQ` — Total magnitude of Lorentz force
3. `TOTPOT` — Total photospheric magnetic free energy density
4. `TOTUSJZ` — Total unsigned vertical current
5. `ABSNJZH` — Absolute value of the net current helicity

The model is created with **default QUANT parameters** using:

```python
quant = QUANTClassifier()
```

Expected input format:

- `X_train_standardized_aeon.npy`
- `X_test_standardized_aeon.npy`
- `y_train.npy`
- `y_test.npy`
- `feature_columns.json`

The expected aeon tensor shape is `(n_cases, n_channels, n_timepoints)`, for example `(277010, 47, 60)`. The notebook also handles the alternate shape `(n_cases, n_timepoints, n_channels)` by transposing after feature selection.


## 1. Install and import packages

Run this cell first. It installs `aeon` only if it is not already available in the environment.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import importlib.util
import subprocess
import sys

# Install aeon only when needed. This keeps the notebook usable in fresh Colab sessions.
if importlib.util.find_spec("aeon") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-U", "aeon", "scikit-learn", "joblib", "pandas"])
else:
    print("aeon is already installed.")


aeon is already installed.


In [ ]:
from pathlib import Path
import gc
import json
import time

import joblib
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from sklearn.ensemble import ExtraTreesClassifier
from aeon.classification.interval_based import QUANTClassifier
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split

print("Imports complete.")

Imports complete.


## 2. Configure paths and selected features

Edit `DATA_DIR` so it points to the folder containing your standardized tensors and label arrays.

The model itself still uses default QUANT parameters. The sample-related settings below only control whether you want to run a quicker test before using the full dataset.


In [ ]:
# =============================
# Main path configuration
# =============================
# Change this to the folder where you saved the standardized dataset.
DATA_DIR = Path("/content/drive/MyDrive/solar_flare_forecasting/model_ready_partition_split/final_clean_magnetic_only")

OUTPUT_DIR = DATA_DIR / "quant_11_feature_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Updated to 11 features (removed EPSZ / index 9 as requested)
FEATURES_TO_USE = [
    "TOTUSJH", "TOTBSQ", "TOTPOT", "TOTUSJZ", "ABSNJZH",
    "SAVNCPP", "USFLUX", "TOTFZ", "MEANPOT", "R_VALUE",
    "SHRGT45"
]

# Optional quick test settings.
TRAIN_SAMPLE_SIZE = None
TEST_SAMPLE_SIZE = None
RANDOM_SEED = 42

# QUANT / aeon validity checks
DROP_LOW_VARIANCE_CASES = True
VARIANCE_THRESHOLD = 1e-7

print(f"DATA_DIR: {DATA_DIR}")
print(f"OUTPUT_DIR: {OUTPUT_DIR}")
print(f"Features ({len(FEATURES_TO_USE)}): {FEATURES_TO_USE}")

DATA_DIR: /content/drive/MyDrive/solar_flare_forecasting/model_ready_partition_split/final_clean_magnetic_only
OUTPUT_DIR: /content/drive/MyDrive/solar_flare_forecasting/model_ready_partition_split/final_clean_magnetic_only/quant_11_feature_outputs
Features (11): ['TOTUSJH', 'TOTBSQ', 'TOTPOT', 'TOTUSJZ', 'ABSNJZH', 'SAVNCPP', 'USFLUX', 'TOTFZ', 'MEANPOT', 'R_VALUE', 'SHRGT45']





 afsd## 3. Locate and load the standardized dataset

This cell tries common filenames used in the standardized dataset notebooks. Adjust the candidate lists only if your files have different names.


In [ ]:
def find_first_existing(data_dir: Path, candidates: list[str]) -> Path:
    """Return the first existing file path from a list of candidate filenames."""
    for name in candidates:
        path = data_dir / name
        if path.exists():
            return path
    candidate_text = "\n".join(str(data_dir / name) for name in candidates)
    raise FileNotFoundError(
        "None of the expected files were found. Checked:\n" + candidate_text
    )

x_train_path = find_first_existing(DATA_DIR, [
    "X_train.npy",
])
x_test_path = find_first_existing(DATA_DIR, [
    "X_test.npy",
])
y_train_path = find_first_existing(DATA_DIR, [
    "y_train.npy",
])
y_test_path = find_first_existing(DATA_DIR, [
    "y_test.npy"
])
feature_cols_path = find_first_existing(DATA_DIR, [
    "feature_columns_final_magnetic.json"
])

print("Found files:")
print("X train:", x_train_path)
print("X test: ", x_test_path)
print("y train:", y_train_path)
print("y test: ", y_test_path)
print("features:", feature_cols_path)

Found files:
X train: /content/drive/MyDrive/solar_flare_forecasting/model_ready_partition_split/final_clean_magnetic_only/X_train.npy
X test:  /content/drive/MyDrive/solar_flare_forecasting/model_ready_partition_split/final_clean_magnetic_only/X_test.npy
y train: /content/drive/MyDrive/solar_flare_forecasting/model_ready_partition_split/final_clean_magnetic_only/y_train.npy
y test:  /content/drive/MyDrive/solar_flare_forecasting/model_ready_partition_split/final_clean_magnetic_only/y_test.npy
features: /content/drive/MyDrive/solar_flare_forecasting/model_ready_partition_split/final_clean_magnetic_only/feature_columns_final_magnetic.json


In [ ]:
# Memory-map the large X arrays first. The selected 5 channels are copied into memory later.
X_train_raw = np.load(x_train_path, mmap_mode="r")
X_test_raw = np.load(x_test_path, mmap_mode="r")
y_train = np.load(y_train_path)
y_test = np.load(y_test_path)

with open(feature_cols_path, "r") as f:
    feature_columns = json.load(f)

print("Raw tensor shapes:")
print("X_train_raw:", X_train_raw.shape, X_train_raw.dtype)
print("X_test_raw: ", X_test_raw.shape, X_test_raw.dtype)
print("y_train:    ", y_train.shape, y_train.dtype)
print("y_test:     ", y_test.shape, y_test.dtype)
print("Number of feature columns:", len(feature_columns))
print("First 10 feature columns:", feature_columns[:10])


Raw tensor shapes:
X_train_raw: (277010, 24, 60) float32
X_test_raw:  (50963, 24, 60) float32
y_train:     (277010,) int8
y_test:      (50963,) int8
Number of feature columns: 24
First 10 feature columns: ['TOTUSJH', 'TOTBSQ', 'TOTPOT', 'TOTUSJZ', 'ABSNJZH', 'SAVNCPP', 'USFLUX', 'TOTFZ', 'MEANPOT', 'EPSZ']


## 4. Select the five requested channels

The aeon input shape should be:

```text
(n_cases, n_channels, n_timepoints)
```

After this step, the selected tensors should have shape:

```text
(n_cases, 5, 60)
```

assuming your standardized windows are 60 time steps long.


In [ ]:
missing_features = [feat for feat in FEATURES_TO_USE if feat not in feature_columns]
if missing_features:
    raise ValueError(f"These requested features are missing from feature_columns.json: {missing_features}")

print("Mapping selected feature indices...")
selected_indices = []
feature_index_map = {}
for feat in tqdm(FEATURES_TO_USE, desc="Features"):
    idx = feature_columns.index(feat)
    selected_indices.append(idx)
    feature_index_map[feat] = idx

print("Selected feature indices:")
for feat, idx in feature_index_map.items():
    print(f"  {feat:8s} -> channel index {idx}")

Mapping selected feature indices...


Features:   0%|          | 0/11 [00:00<?, ?it/s]

Selected feature indices:
  TOTUSJH  -> channel index 0
  TOTBSQ   -> channel index 1
  TOTPOT   -> channel index 2
  TOTUSJZ  -> channel index 3
  ABSNJZH  -> channel index 4
  SAVNCPP  -> channel index 5
  USFLUX   -> channel index 6
  TOTFZ    -> channel index 7
  MEANPOT  -> channel index 8
  R_VALUE  -> channel index 23
  SHRGT45  -> channel index 11


In [ ]:
def select_channels_as_aeon(X, selected_indices, n_total_features):
    """Select requested channels and return shape (n_cases, n_selected_channels, n_timepoints)."""
    if X.ndim != 3:
        raise ValueError(f"Expected a 3D tensor, but got shape {X.shape}")

    # Preferred aeon layout: (n_cases, n_channels, n_timepoints)
    if X.shape[1] == n_total_features:
        X_selected = X[:, selected_indices, :]
        layout = "channels_second / aeon format"

    # Alternate layout: (n_cases, n_timepoints, n_channels)
    elif X.shape[2] == n_total_features:
        X_selected = X[:, :, selected_indices].transpose(0, 2, 1)
        layout = "channels_last -> transposed to aeon format"

    else:
        raise ValueError(
            f"Could not identify channel axis. X shape is {X.shape}, "
            f"but feature_columns has length {n_total_features}."
        )

    # Convert to float32 to keep memory lower and match the saved standardized tensors.
    return np.asarray(X_selected, dtype=np.float32), layout

X_train_5, train_layout = select_channels_as_aeon(X_train_raw, selected_indices, len(feature_columns))
X_test_5, test_layout = select_channels_as_aeon(X_test_raw, selected_indices, len(feature_columns))

print("Train layout:", train_layout)
print("Test layout: ", test_layout)
print("X_train_5:", X_train_5.shape, X_train_5.dtype)
print("X_test_5: ", X_test_5.shape, X_test_5.dtype)

# Free raw memory-map handles if they are no longer needed.
del X_train_raw, X_test_raw
gc.collect()


Train layout: channels_second / aeon format
Test layout:  channels_second / aeon format
X_train_5: (277010, 11, 60) float32
X_test_5:  (50963, 11, 60) float32


19

## 5. Basic dataset checks

This verifies:

- `X` and `y` have matching case counts
- the tensors contain finite values
- class counts are visible before training


In [ ]:
def print_class_counts(y, name):
    values, counts = np.unique(y, return_counts=True)
    df = pd.DataFrame({"class": values, "count": counts})
    df["percent"] = 100 * df["count"] / len(y)
    print(f"\n{name} class counts:")
    display(df)

if X_train_5.shape[0] != len(y_train):
    raise ValueError(f"Train case mismatch: X has {X_train_5.shape[0]} cases, y has {len(y_train)}")
if X_test_5.shape[0] != len(y_test):
    raise ValueError(f"Test case mismatch: X has {X_test_5.shape[0]} cases, y has {len(y_test)}")

if not np.isfinite(X_train_5).all():
    raise ValueError("X_train_5 contains NaN or infinite values. QUANT cannot handle missing values.")
if not np.isfinite(X_test_5).all():
    raise ValueError("X_test_5 contains NaN or infinite values. QUANT cannot handle missing values.")

print("Shape checks passed.")
print_class_counts(y_train, "Train")
print_class_counts(y_test, "Test")


Shape checks passed.

Train class counts:


,class,count,percent
0,0,271941,98.170102
1,1,5069,1.829898



Test class counts:


,class,count,percent
0,0,49810,97.737574
1,1,1153,2.262426


## 6. Optional stratified sampling

Leave `TRAIN_SAMPLE_SIZE = None` and `TEST_SAMPLE_SIZE = None` to use the full dataset.

Sampling is useful only for a quick smoke test before running QUANT on the full standardized dataset.


In [ ]:
def stratified_sample(X, y, sample_size, random_seed=42):
    """Return a stratified subset of X and y. If sample_size is None, return all cases."""
    if sample_size is None or sample_size >= len(y):
        return X, y, np.arange(len(y))

    indices = np.arange(len(y))
    sample_idx, _ = train_test_split(
        indices,
        train_size=sample_size,
        stratify=y,
        random_state=random_seed,
    )
    sample_idx = np.sort(sample_idx)
    return X[sample_idx], y[sample_idx], sample_idx

X_train_fit, y_train_fit, train_used_idx = stratified_sample(
    X_train_5, y_train, TRAIN_SAMPLE_SIZE, RANDOM_SEED
)
X_test_eval, y_test_eval, test_used_idx = stratified_sample(
    X_test_5, y_test, TEST_SAMPLE_SIZE, RANDOM_SEED
)

print("Training set used:", X_train_fit.shape, y_train_fit.shape)
print("Test set used:    ", X_test_eval.shape, y_test_eval.shape)
print_class_counts(y_train_fit, "Train used")
print_class_counts(y_test_eval, "Test used")


Training set used: (277010, 11, 60) (277010,)
Test set used:     (50963, 11, 60) (50963,)

Train used class counts:


,class,count,percent
0,0,271941,98.170102
1,1,5069,1.829898



Test used class counts:


,class,count,percent
0,0,49810,97.737574
1,1,1153,2.262426


## 7. Check for low-variance case/channel pairs

aeon's collection validation can raise an error when any individual case/channel has too little variation. This is separate from model hyperparameters; it is an input-data validity check.

By default, this notebook removes affected cases from train and test before calling QUANT. The removed counts are printed so the change is transparent.


In [ ]:
def remove_low_variance_cases(X, y, threshold=1e-7, name="dataset"):
    """Remove cases where any selected channel has std <= threshold across time."""
    # Since np.std over the whole axis is fast, we wrap the logic in a simple progress-monitored step
    print(f"Calculating variance for {name}...")
    per_case_channel_std = np.std(X, axis=2)

    keep_mask = []
    for row in tqdm(per_case_channel_std, desc=f"Checking {name} cases"):
        keep_mask.append((row > threshold).all())

    keep_mask = np.array(keep_mask)
    n_removed = int((~keep_mask).sum())

    print(f"{name}: {n_removed} / {len(y)} cases have at least one low-variance selected channel.")

    if n_removed > 0:
        bad_pairs = np.argwhere(per_case_channel_std <= threshold)
        preview = bad_pairs[:10]
        print("First low-variance case/channel pairs shown as [case_index, selected_channel_index]:")
        print(preview)
        print("Selected channel order:", FEATURES_TO_USE)

    return X[keep_mask], y[keep_mask], keep_mask

if DROP_LOW_VARIANCE_CASES:
    X_train_fit, y_train_fit, train_keep_mask = remove_low_variance_cases(
        X_train_fit, y_train_fit, VARIANCE_THRESHOLD, "train"
    )
    X_test_eval, y_test_eval, test_keep_mask = remove_low_variance_cases(
        X_test_eval, y_test_eval, VARIANCE_THRESHOLD, "test"
    )
else:
    print("Low-variance cases were not removed because DROP_LOW_VARIANCE_CASES = False.")

print("Final training shape:", X_train_fit.shape, y_train_fit.shape)
print("Final test shape:    ", X_test_eval.shape, y_test_eval.shape)
print_class_counts(y_train_fit, "Final train")
print_class_counts(y_test_eval, "Final test")

Calculating variance for train...


Checking train cases:   0%|          | 0/277010 [00:00<?, ?it/s]

train: 109227 / 277010 cases have at least one low-variance selected channel.
First low-variance case/channel pairs shown as [case_index, selected_channel_index]:
[[1525    9]
 [1793    9]
 [1794    9]
 [1795    9]
 [1796    9]
 [1797    9]
 [1798    9]
 [1799    9]
 [1800    9]
 [1801    9]]
Selected channel order: ['TOTUSJH', 'TOTBSQ', 'TOTPOT', 'TOTUSJZ', 'ABSNJZH', 'SAVNCPP', 'USFLUX', 'TOTFZ', 'MEANPOT', 'R_VALUE', 'SHRGT45']
Calculating variance for test...


Checking test cases:   0%|          | 0/50963 [00:00<?, ?it/s]

test: 20934 / 50963 cases have at least one low-variance selected channel.
First low-variance case/channel pairs shown as [case_index, selected_channel_index]:
[[1272    9]
 [1273    9]
 [1274    9]
 [1275    9]
 [1276    9]
 [1277    9]
 [1278    9]
 [1279    9]
 [1280    9]
 [1281    9]]
Selected channel order: ['TOTUSJH', 'TOTBSQ', 'TOTPOT', 'TOTUSJZ', 'ABSNJZH', 'SAVNCPP', 'USFLUX', 'TOTFZ', 'MEANPOT', 'R_VALUE', 'SHRGT45']
Final training shape: (167783, 11, 60) (167783,)
Final test shape:     (30029, 11, 60) (30029,)

Final train class counts:


,class,count,percent
0,0,162716,96.980028
1,1,5067,3.019972



Final test class counts:


,class,count,percent
0,0,28876,96.160378
1,1,1153,3.839622


## 8. Train the default QUANT model

This is the key model cell. No custom parameters are passed to `QUANTClassifier`, so aeon's default parameters are used.


In [ ]:
quant = QUANTClassifier(
    estimator=ExtraTreesClassifier(
        n_estimators=200,
        n_jobs=-1,
        random_state=42
    ),
    random_state=42
)
print("Default QUANT parameters:")
print(quant.get_params())

start_time = time.time()
quant.fit(X_train_fit, y_train_fit)
fit_seconds = time.time() - start_time

print(f"QUANT fit time: {fit_seconds / 60:.2f} minutes ({fit_seconds:.1f} seconds)")


Default QUANT parameters:
{'class_weight': None, 'estimator__bootstrap': False, 'estimator__ccp_alpha': 0.0, 'estimator__class_weight': None, 'estimator__criterion': 'gini', 'estimator__max_depth': None, 'estimator__max_features': 'sqrt', 'estimator__max_leaf_nodes': None, 'estimator__max_samples': None, 'estimator__min_impurity_decrease': 0.0, 'estimator__min_samples_leaf': 1, 'estimator__min_samples_split': 2, 'estimator__min_weight_fraction_leaf': 0.0, 'estimator__monotonic_cst': None, 'estimator__n_estimators': 200, 'estimator__n_jobs': -1, 'estimator__oob_score': False, 'estimator__random_state': 42, 'estimator__verbose': 0, 'estimator__warm_start': False, 'estimator': ExtraTreesClassifier(n_estimators=200, n_jobs=-1, random_state=42), 'interval_depth': 6, 'quantile_divisor': 4, 'random_state': 42}


## 9. Predict on the test set


In [ ]:
start_time = time.time()
y_pred = quant.predict(X_test_eval)
predict_seconds = time.time() - start_time

print(f"Prediction time: {predict_seconds / 60:.2f} minutes ({predict_seconds:.1f} seconds)")
print("Predictions shape:", y_pred.shape)


In [ ]:
# QUANT supports probability predictions through aeon/sklearn-style predict_proba.
y_proba = None
try:
    y_proba = quant.predict_proba(X_test_eval)
    print("Predicted probabilities shape:", y_proba.shape)
    print("Class order:", quant.classes_)
except Exception as exc:
    print("predict_proba was not available or failed:", repr(exc))


## 10. Evaluate performance

For the solar flare task, raw accuracy can be misleading because the classes are imbalanced. This notebook reports:

- accuracy
- balanced accuracy
- precision
- recall / POD
- F1
- TSS
- HSS
- ROC-AUC and PR-AUC when probabilities are available


In [ ]:
def binary_skill_scores(y_true, y_pred, positive_label=1):
    """Compute common flare-forecasting binary skill scores."""
    y_true_pos = np.asarray(y_true) == positive_label
    y_pred_pos = np.asarray(y_pred) == positive_label

    tn, fp, fn, tp = confusion_matrix(
        y_true_pos,
        y_pred_pos,
        labels=[False, True],
    ).ravel()

    pod = tp / (tp + fn) if (tp + fn) else np.nan          # Probability of detection / recall
    far = fp / (tp + fp) if (tp + fp) else np.nan          # False alarm ratio
    fpr = fp / (fp + tn) if (fp + tn) else np.nan          # False positive rate
    tss = pod - fpr if np.isfinite(pod) and np.isfinite(fpr) else np.nan

    hss_denom = ((tp + fn) * (fn + tn)) + ((tp + fp) * (fp + tn))
    hss = (2 * ((tp * tn) - (fp * fn)) / hss_denom) if hss_denom else np.nan

    return {
        "TP": int(tp),
        "TN": int(tn),
        "FP": int(fp),
        "FN": int(fn),
        "POD_recall": pod,
        "FAR": far,
        "FPR": fpr,
        "TSS": tss,
        "HSS": hss,
    }

unique_labels = np.unique(np.concatenate([y_train_fit, y_test_eval]))
positive_label = 1 if 1 in unique_labels else unique_labels[-1]
print("Using positive label:", positive_label)

metrics = {
    "accuracy": accuracy_score(y_test_eval, y_pred),
    "balanced_accuracy": balanced_accuracy_score(y_test_eval, y_pred),
    "precision_positive": precision_score(y_test_eval, y_pred, pos_label=positive_label, zero_division=0),
    "recall_positive": recall_score(y_test_eval, y_pred, pos_label=positive_label, zero_division=0),
    "f1_positive": f1_score(y_test_eval, y_pred, pos_label=positive_label, zero_division=0),
    "fit_seconds": fit_seconds,
    "predict_seconds": predict_seconds,
    "n_train_cases": int(len(y_train_fit)),
    "n_test_cases": int(len(y_test_eval)),
    "features_used": FEATURES_TO_USE,
    "model": "aeon.classification.interval_based.QUANTClassifier",
    "model_params": quant.get_params(),
}
metrics.update(binary_skill_scores(y_test_eval, y_pred, positive_label=positive_label))

if y_proba is not None:
    class_list = list(quant.classes_)
    if positive_label in class_list:
        pos_col = class_list.index(positive_label)
        positive_scores = y_proba[:, pos_col]
        metrics["roc_auc"] = roc_auc_score(y_test_eval == positive_label, positive_scores)
        metrics["average_precision_pr_auc"] = average_precision_score(y_test_eval == positive_label, positive_scores)

metrics_df = pd.DataFrame([metrics]).T
metrics_df.columns = ["value"]
display(metrics_df)

print("Classification report:")
print(classification_report(y_test_eval, y_pred, zero_division=0))

cm_labels = np.unique(np.concatenate([y_test_eval, y_pred]))
cm = confusion_matrix(y_test_eval, y_pred, labels=cm_labels)
cm_df = pd.DataFrame(
    cm,
    index=[f"true_{label}" for label in cm_labels],
    columns=[f"pred_{label}" for label in cm_labels],
)
display(cm_df)


## 11. Save predictions, metrics, and the fitted model

The model file can be large. Set `SAVE_MODEL = False` if you only want metrics and predictions.


In [ ]:
SAVE_MODEL = True

predictions_df = pd.DataFrame({
    "y_true": y_test_eval,
    "y_pred": y_pred,
})

if y_proba is not None:
    print("Adding probabilities to dataframe...")
    for i, cls in enumerate(tqdm(quant.classes_, desc="Classes")):
        predictions_df[f"proba_class_{cls}"] = y_proba[:, i]

predictions_path = OUTPUT_DIR / "quant_default_5_features_predictions.csv"
metrics_path = OUTPUT_DIR / "quant_default_5_features_metrics.json"
model_path = OUTPUT_DIR / "quant_default_5_features_model.joblib"

print("Writing predictions CSV...")
predictions_df.to_csv(predictions_path, index=False)

# Convert any numpy objects to regular Python objects for JSON saving.
def make_json_safe(obj):
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, list):
        return [make_json_safe(x) for x in obj]
    if isinstance(obj, dict):
        return {str(k): make_json_safe(v) for k, v in obj.items()}
    # Convert any other non-serializable objects to string representation
    if not isinstance(obj, (str, int, float, bool, type(None))):
        return str(obj)
    return obj

with open(metrics_path, "w") as f:
    json.dump(make_json_safe(metrics), f, indent=2)

print("Saved predictions to:", predictions_path)
print("Saved metrics to:    ", metrics_path)

if SAVE_MODEL:
    print("Serializing model (this may take a moment)...")
    joblib.dump(quant, model_path)
    print("Saved fitted model to:", model_path)
else:
    print("SAVE_MODEL is False, so the fitted model was not saved.")

## 12. Notes for running on the full dataset

- To train on the full standardized dataset, keep:

```python
TRAIN_SAMPLE_SIZE = None
TEST_SAMPLE_SIZE = None
```

- QUANT is a CPU model. A GPU or TPU will usually not help because this aeon classifier is not a neural network model.
- The model line should remain `QUANTClassifier()` if you want aeon's default parameters.
- If aeon raises a low-variation error, keep `DROP_LOW_VARIANCE_CASES = True` and rerun from the low-variance check cell onward.
- For flare prediction, prioritize **balanced accuracy, TSS, HSS, recall/POD, and false alarms** over raw accuracy because the positive class is much rarer than the negative class.
